# 12A — พื้นฐานการพล็อตแผนที่อากาศ

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**บทเสริม · ใช้เวลาประมาณ 2 ชั่วโมง · เรียนก่อนบทที่ 12**

---

## ทำไมต้องมีบทนี้

บทที่ 12 เรียกฟังก์ชันสำเร็จรูปอย่าง `eu.shade()` กับ `ec.panel_daily()`
ซึ่งได้ผลเร็วก็จริง แต่ไม่เห็นว่าข้างในทำอะไร พอต้องทำแผนที่ของตัวเองจะติดทันที

**บทนี้ไม่ใช้ฟังก์ชันสำเร็จรูปเลยแม้แต่ตัวเดียว** เรียก `matplotlib` กับ `cartopy` ตรง ๆ ทุกบรรทัด
สร้างแผนที่ทีละชั้นจากกระดาษเปล่า จนได้แผนที่อากาศที่ใช้ในรายงานได้จริง

เรียนจบบทนี้แล้วจะอ่านโค้ดในบทที่ 12 ออกทั้งหมด และแก้ให้เป็นแบบที่ตัวเองต้องการได้

---

### สิ่งที่จะได้

- รู้ว่าแผนที่อากาศหนึ่งใบประกอบด้วย **ชั้นข้อมูล** อะไรบ้าง และวางทับกันอย่างไร
- เข้าใจ **projection** กับ **transform** ว่าต่างกันตรงไหน (จุดที่ผิดกันมากที่สุด)
- ใช้ `contourf`, `contour`, `clabel`, `barbs`, `quiver`, `streamplot` ได้คล่อง
- เลือก **colormap** และ **levels** ให้สื่อความหมายถูกต้อง
- รู้จักข้อผิดพลาดที่เจอบ่อย 7 ข้อ และวิธีเลี่ยง

**ต้องผ่านบทที่ 10 (Matplotlib) มาก่อน**

## 0. เตรียมเครื่องมือและข้อมูล

In [ ]:
!pip install -q cartopy
print("ติดตั้ง cartopy เรียบร้อย")

In [ ]:
import os, urllib.request

BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

os.makedirs("data/cases", exist_ok=True)
NC = "data/cases/dianmu2021.nc"
if not os.path.exists(NC):
    urllib.request.urlretrieve(BASE + "data/cases/dianmu2021.nc", NC)

# โหลดฟอนต์ไทยไว้ใช้กับ matplotlib
import matplotlib, matplotlib.font_manager as fm
try:
    if not os.path.exists("Sarabun-Regular.ttf"):
        urllib.request.urlretrieve(
            "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf",
            "Sarabun-Regular.ttf")
    fm.fontManager.addfont("Sarabun-Regular.ttf")
    matplotlib.rc("font", family="Sarabun")
    matplotlib.rcParams["axes.unicode_minus"] = False
    print("ฟอนต์ไทยพร้อมใช้งาน")
except Exception as exc:
    print("ตั้งฟอนต์ไทยไม่สำเร็จ:", exc, "| ใช้ label ภาษาอังกฤษแทนได้")

print("ไฟล์ข้อมูล:", os.path.getsize(NC) / 1024 / 1024, "MB")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams["figure.dpi"] = 110

ds = xr.open_dataset("data/cases/dianmu2021.nc")

# ERA5 เรียง latitude จากมากไปน้อย เรียงใหม่ให้เป็นน้อยไปมากก่อน
# ไม่งั้น slice(5, 21) จะได้ข้อมูลเปล่า
if float(ds["latitude"][0]) > float(ds["latitude"][-1]):
    ds = ds.sortby("latitude")

print("มิติ  :", dict(ds.sizes))
print("ตัวแปร:", list(ds.data_vars))
print("เวลา  :", str(ds["time"].values[0])[:16], "ถึง", str(ds["time"].values[-1])[:16])

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 1. แผนที่อากาศประกอบด้วยอะไรบ้าง

แผนที่อากาศคือการวาง **ชั้นข้อมูล** ซ้อนกัน เรียงจากล่างขึ้นบน

| ลำดับ | ชั้น | คำสั่ง | ใช้แสดง |
|---|---|---|---|
| 1 | แผนที่ฐาน | `add_feature` | ชายฝั่ง พรมแดน พื้นดิน ทะเล |
| 2 | สนามแรเงา | `contourf` | ปริมาณต่อเนื่อง 1 ตัว เช่น อุณหภูมิ ฝน |
| 3 | เส้นชั้น | `contour` + `clabel` | ปริมาณอีกตัว เช่น ความกดอากาศ |
| 4 | ลม | `barbs` / `quiver` / `streamplot` | ทิศทางและความเร็ว |
| 5 | จุดและข้อความ | `scatter` / `text` | สถานี เมือง ศูนย์กลางพายุ |
| 6 | ส่วนอธิบาย | `colorbar` / `set_title` | หน่วย เวลา แหล่งข้อมูล |

**กฎสำคัญ: หนึ่งแผนที่ควรมีสนามข้อมูลไม่เกิน 3 ชั้น** (แรเงา 1 + เส้นชั้น 1 + ลม 1)
มากกว่านี้คนอ่านจะแยกไม่ออกว่าอะไรเป็นอะไร

ลองนึกถึงแผนที่อากาศที่เห็นในข่าว — มีแค่เส้นความกดอากาศกับสัญลักษณ์ไม่กี่ตัวเท่านั้น

## 2. ชั้นที่ 1 — แผนที่ฐาน

เริ่มจากกระดาษเปล่าก่อน ยังไม่มีข้อมูลอะไรเลย

In [ ]:
fig = plt.figure(figsize=(9, 7))

# ต้องบอก projection ตอนสร้างแกน จะเพิ่มทีหลังไม่ได้
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# ขอบเขตแผนที่: [ตะวันตก, ตะวันออก, ใต้, เหนือ]
ax.set_extent([95, 115, 5, 25], crs=ccrs.PlateCarree())

# cartopy ไปโหลดข้อมูลเส้นชายฝั่งจากเว็บ Natural Earth ตอนวาดรูป
# จึงครอบ try ไว้ เผื่อเครือข่ายบล็อก (อธิบายละเอียดในหัวข้อถัดไป)
try:
    next(iter(cfeature.COASTLINE.geometries()), None)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
except Exception as exc:
    print("โหลดเส้นชายฝั่งไม่ได้ ->", exc)

plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### เติมรายละเอียดให้แผนที่ฐาน

`cfeature` มีชั้นข้อมูลภูมิศาสตร์สำเร็จรูปให้ใช้

| ชื่อ | คือ |
|---|---|
| `COASTLINE` | เส้นชายฝั่ง |
| `BORDERS` | พรมแดนประเทศ |
| `LAND` / `OCEAN` | พื้นดิน / ทะเล (ใช้ระบายสีพื้น) |
| `LAKES` / `RIVERS` | ทะเลสาบ / แม่น้ำ |

⚠️ cartopy **ไม่ได้ติดตั้งข้อมูลพวกนี้มาด้วย** แต่ไปโหลดจากเว็บ Natural Earth ตอนวาดรูป
ถ้าเครือข่ายบล็อก จะพังตอน `plt.show()` ซึ่งดักยาก จึงควรบังคับโหลดก่อนด้วย `try`

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([95, 115, 5, 25], crs=ccrs.PlateCarree())

try:
    # บังคับให้ cartopy โหลดข้อมูลตอนนี้ จะได้ดัก error ได้ทัน
    for feat in (cfeature.COASTLINE, cfeature.BORDERS, cfeature.OCEAN, cfeature.LAND):
        next(iter(feat.geometries()), None)

    ax.add_feature(cfeature.OCEAN, facecolor="#eaf3fb", zorder=0)
    ax.add_feature(cfeature.LAND, facecolor="#f7f5f0", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
except Exception as exc:
    print("โหลดแผนที่ฐานไม่ได้ ->", exc)

# เส้นกริดพร้อมตัวเลขพิกัด
gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                  alpha=0.5, linestyle="--")
gl.top_labels = False      # ไม่เอาตัวเลขด้านบน
gl.right_labels = False    # ไม่เอาตัวเลขด้านขวา

ax.set_title("แผนที่ฐาน — ภูมิภาคอินโดจีน", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

> **`zorder`** คือลำดับการวางทับ ตัวเลขน้อยอยู่ล่าง มากอยู่บน
> ให้สีพื้นดิน/ทะเลเป็น `zorder=0` เสมอ ไม่งั้นมันจะไปทับข้อมูลที่พล็อตทีหลัง

## 2.5 เขียนฟังก์ชันของตัวเองไว้ใช้ซ้ำ

โค้ด 15 บรรทัดข้างบนจะต้องพิมพ์ซ้ำทุกครั้งที่ทำแผนที่ใหม่ ซึ่งไม่คุ้ม
นี่คือจังหวะที่ควรใช้ `def` ตามที่เรียนมาในบทที่ 07

ฟังก์ชันข้างล่างนี้ไม่ได้มาจากไหน — เราเขียนเองทุกบรรทัด จากโค้ดที่เพิ่งทำมา

In [ ]:
def add_basemap(ax, extent, ocean=True, land=True, gridlines=True):
    """
    เตรียมแผนที่ฐานให้แกนที่ส่งเข้ามา

    ax        : แกนที่สร้างด้วย projection=ccrs.PlateCarree() แล้ว
    extent    : [ตะวันตก, ตะวันออก, ใต้, เหนือ]
    ocean/land: ระบายสีพื้นทะเลและพื้นดินหรือไม่
    gridlines : วาดเส้นกริดพร้อมตัวเลขพิกัดหรือไม่

    คืนค่า ax ตัวเดิม เพื่อให้เขียนต่อแบบลูกโซ่ได้
    """
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    try:
        # cartopy โหลดข้อมูล Natural Earth ตอนวาดรูป ไม่ใช่ตอนสั่ง
        # ถ้าปล่อยไว้ เครือข่ายที่บล็อกจะทำให้พังตอน plt.show() ซึ่งดักยาก
        # จึงบังคับโหลดตรงนี้ เพื่อให้ except ทำงานทัน
        for feat in (cfeature.COASTLINE, cfeature.BORDERS,
                     cfeature.OCEAN, cfeature.LAND):
            next(iter(feat.geometries()), None)

        if ocean:
            ax.add_feature(cfeature.OCEAN, facecolor="#eaf3fb", zorder=0)
        if land:
            ax.add_feature(cfeature.LAND, facecolor="#f7f5f0", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=":")
    except Exception as exc:
        print("โหลดแผนที่ฐานไม่ได้ ->", exc)
        print("แผนที่จะไม่มีเส้นชายฝั่ง แต่ข้อมูลที่พล็อตยังถูกต้องทุกอย่าง")

    if gridlines:
        gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                          alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False

    return ax


# ทดสอบว่าใช้ได้จริง
fig = plt.figure(figsize=(8, 6.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [95, 115, 5, 25])
ax.set_title("แผนที่ฐานจากฟังก์ชันที่เขียนเอง", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

จากนี้ไปจะใช้ `add_basemap()` แทนการพิมพ์ซ้ำ โค้ดแต่ละเซลล์จะสั้นลงมาก
และโฟกัสไปที่ **ชั้นข้อมูล** ซึ่งเป็นสิ่งที่เรากำลังเรียน

> นี่คือหลักการเดียวกับที่ `utils/era5_utils.py` ใช้ ต่างกันแค่ว่าในไฟล์นั้น
> ผมเขียนไว้ให้แล้ว ส่วนตรงนี้เราเขียนเอง — พอเข้าใจแล้วจะแก้ของใครก็ได้

## 3. projection กับ transform — จุดที่ผิดกันมากที่สุด

สองคำนี้ต่างกันคนละเรื่อง แต่ชื่อคล้ายกันจนสับสน

| | คืออะไร | ใส่ตรงไหน |
|---|---|---|
| **projection** | ระบบพิกัดที่จะ **วาดออกมา** | ตอนสร้างแกน `add_subplot(..., projection=...)` |
| **transform** | ระบบพิกัดที่ **ข้อมูลอยู่** | ทุกคำสั่งพล็อต `contourf(..., transform=...)` |

ข้อมูล ERA5 เป็นละติจูด–ลองจิจูดธรรมดา ดังนั้น **`transform=ccrs.PlateCarree()` เสมอ**
ส่วน projection จะเลือกเป็นอะไรก็ได้ตามต้องการ

ถ้าลืมใส่ `transform` cartopy จะเดาว่าข้อมูลอยู่ในระบบเดียวกับแกน
ซึ่งบังเอิญถูกถ้า projection เป็น PlateCarree — **แต่พอเปลี่ยน projection เมื่อไรแผนที่จะเพี้ยนทันที**

ลองดูตัวอย่างข้างล่าง แผนที่ซ้ายถูก แผนที่ขวาลืม `transform`

In [ ]:
t2m = ds["t2m"].sel(time="2021-09-24T12:00") - 273.15

fig = plt.figure(figsize=(14, 5.5))

# --- ซ้าย: ใส่ transform ถูกต้อง ---
ax1 = fig.add_subplot(1, 2, 1, projection=ccrs.LambertConformal(
    central_longitude=105, central_latitude=15))
ax1.set_extent([95, 115, 5, 25], crs=ccrs.PlateCarree())
try:
    next(iter(cfeature.COASTLINE.geometries()), None)
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
except Exception:
    pass
ax1.contourf(t2m["longitude"], t2m["latitude"], t2m.values,
             levels=np.arange(18, 34, 1), cmap="RdYlBu_r",
             transform=ccrs.PlateCarree())
ax1.set_title("ถูก — ใส่ transform=PlateCarree()", loc="left")

# --- ขวา: ลืม transform ---
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.LambertConformal(
    central_longitude=105, central_latitude=15))
ax2.set_extent([95, 115, 5, 25], crs=ccrs.PlateCarree())
try:
    next(iter(cfeature.COASTLINE.geometries()), None)
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.8)
except Exception:
    pass
ax2.contourf(t2m["longitude"], t2m["latitude"], t2m.values,
             levels=np.arange(18, 34, 1), cmap="RdYlBu_r")
ax2.set_title("ผิด — ลืม transform ข้อมูลไปอยู่ผิดที่", loc="left")

plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

เห็นความต่างชัดเจน — รูปขวาข้อมูลไปกองผิดตำแหน่ง เพราะ cartopy ตีความตัวเลข
ลองจิจูด–ละติจูดว่าเป็นหน่วยเมตรของระบบ Lambert

**จำให้ขึ้นใจ: ทุกคำสั่งที่วาดข้อมูลลงแผนที่ ต้องมี `transform=ccrs.PlateCarree()`**
ไม่ว่าจะเป็น `contourf`, `contour`, `barbs`, `scatter`, `text`, `plot`

## 4. ชั้นที่ 2 — สนามแรเงาด้วย `contourf`

`contourf(x, y, values, levels, cmap, transform)` ระบายสีพื้นตามค่าข้อมูล

- `x`, `y` คือพิกัด 1 มิติ (ลองจิจูด ละติจูด)
- `values` คืออาร์เรย์ 2 มิติ ขนาด (len(y), len(x))
- `levels` คือขอบของแต่ละช่วงสี
- `extend` บอกวิธีจัดการค่าที่เกินช่วง: `"both"`, `"max"`, `"min"`, `"neither"`

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [95, 115, 5, 25])

cf = ax.contourf(t2m["longitude"], t2m["latitude"], t2m.values,
                 levels=np.arange(18, 34.1, 1),
                 cmap="RdYlBu_r",
                 extend="both",
                 transform=ccrs.PlateCarree())

cb = plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.03)
cb.set_label("อุณหภูมิที่ 2 เมตร (°C)")

ax.set_title("ERA5 — อุณหภูมิ 2 เมตร\n2021-09-24 12 UTC", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### `levels` มีผลต่อการอ่านมากกว่าที่คิด

ลองเทียบสามแบบ — ข้อมูลชุดเดียวกันทั้งหมด

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5),
                         subplot_kw={"projection": ccrs.PlateCarree()})

settings = [
    (5, "levels น้อยเกินไป — หยาบ มองไม่เห็นรายละเอียด"),
    (np.arange(18, 34.1, 1), "พอดี — เห็นโครงสร้างชัด"),
    (np.arange(18, 34.1, 0.2), "ละเอียดเกินไป — ลายตา แยกสีไม่ออก"),
]

for ax, (lv, title) in zip(axes, settings):
    add_basemap(ax, [95, 115, 5, 25], gridlines=False)
    cf = ax.contourf(t2m["longitude"], t2m["latitude"], t2m.values,
                     levels=lv, cmap="RdYlBu_r", extend="both",
                     transform=ccrs.PlateCarree())
    ax.set_title(title, fontsize=9, loc="left")

fig.colorbar(cf, ax=axes, orientation="horizontal", shrink=0.4,
             pad=0.05, label="°C")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

**หลักการเลือก levels**

- เริ่มที่ **8–15 ช่วง** กำลังดีสำหรับสายตาคน
- ใช้ตัวเลขกลม ๆ (1, 2, 5, 10, 20, 50) ไม่ใช่ 0.7 หรือ 3.3
- ถ้าข้อมูลเบ้มาก เช่นฝน ให้ใช้ช่วงไม่เท่ากัน — `[1, 5, 10, 20, 40, 80, 160]`
  เพราะฝน 0 กับ 5 mm ต่างกันมากกว่า 100 กับ 105 mm

### เลือก colormap ให้ถูกชนิด

| ชนิดข้อมูล | ใช้ colormap | ตัวอย่าง |
|---|---|---|
| มีทิศทางเดียว (ฝน ความเร็วลม) | **sequential** | `Blues`, `YlGnBu`, `GnBu`, `viridis` |
| มีค่ากลางที่มีความหมาย (anomaly) | **diverging** | `RdBu_r`, `coolwarm`, `BrBG` |
| หมวดหมู่ | **qualitative** | `tab10`, `Set2` |

**เลี่ยง `jet` และ `rainbow`** — ตาคนเห็นแถบเหลืองสว่างเป็นขอบปลอมที่ไม่มีอยู่จริงในข้อมูล
และคนตาบอดสีอ่านไม่ได้เลย วารสารวิชาการหลายฉบับปฏิเสธรูปที่ใช้ jet แล้ว

In [ ]:
rain = ds["tp"].sel(time="2021-09-24T12:00")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5),
                         subplot_kw={"projection": ccrs.PlateCarree()})
levels = [1, 2, 5, 10, 20, 40, 60]

for ax, (cmap, title) in zip(axes, [
        ("GnBu", "ถูก — sequential เหมาะกับฝน"),
        ("RdBu_r", "ไม่เหมาะ — diverging ทำให้ดูเหมือนมีค่าลบ"),
        ("jet", "แย่ — jet สร้างขอบปลอม ตาบอดสีอ่านไม่ได้")]):
    add_basemap(ax, [95, 115, 5, 25], gridlines=False)
    cf = ax.contourf(rain["longitude"], rain["latitude"], rain.values,
                     levels=levels, cmap=cmap, extend="max",
                     transform=ccrs.PlateCarree())
    plt.colorbar(cf, ax=ax, shrink=0.7)
    ax.set_title(title, fontsize=9, loc="left")

plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 5. ชั้นที่ 3 — เส้นชั้นด้วย `contour`

`contour` วาดเฉพาะเส้น ไม่ระบายสี ใช้ซ้อนทับสนามแรเงาเพื่อแสดงข้อมูลตัวที่สอง
`clabel` เติมตัวเลขกำกับบนเส้น

ธรรมเนียมของแผนที่อากาศผิวพื้นคือวาด isobar ทุก **4 hPa**

In [ ]:
mslp = ds["msl"].sel(time="2021-09-24T12:00") / 100   # Pa -> hPa

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [95, 115, 5, 25])

cs = ax.contour(mslp["longitude"], mslp["latitude"], mslp.values,
                levels=np.arange(980, 1021, 4),
                colors="black", linewidths=1.0,
                transform=ccrs.PlateCarree())

# ตัวเลขกำกับเส้น: inline=True คือเจาะช่องว่างบนเส้นให้ตัวเลข
ax.clabel(cs, inline=True, fontsize=9, fmt="%d")

ax.set_title("ความกดอากาศที่ระดับน้ำทะเล (hPa)\n2021-09-24 12 UTC", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ซ้อนสองชั้น: แรเงา + เส้นชั้น

นี่คือหน้าตาของแผนที่อากาศจริงที่ใช้งานกัน

In [ ]:
wind = np.sqrt(ds["u"].sel(time="2021-09-24T12:00", level=850)**2 +
               ds["v"].sel(time="2021-09-24T12:00", level=850)**2)

fig = plt.figure(figsize=(9.5, 7.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [95, 115, 5, 25])

# ชั้นแรเงา
cf = ax.contourf(wind["longitude"], wind["latitude"], wind.values,
                 levels=np.arange(0, 26, 2), cmap="YlGnBu", extend="max",
                 transform=ccrs.PlateCarree())

# ชั้นเส้นชั้น
cs = ax.contour(mslp["longitude"], mslp["latitude"], mslp.values,
                levels=np.arange(980, 1021, 4), colors="black",
                linewidths=0.9, transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=8, fmt="%d")

cb = plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.03)
cb.set_label("ความเร็วลม 850 hPa (m s$^{-1}$)")

gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4, linestyle="--")
gl.top_labels = gl.right_labels = False

ax.set_title("ความเร็วลม 850 hPa (แรเงา) และความกดอากาศผิวพื้น (เส้น)\n"
             "2021-09-24 12 UTC | ข้อมูล ERA5", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 6. ชั้นที่ 4 — ลม

มี 3 วิธี เลือกตามสิ่งที่อยากสื่อ

| คำสั่ง | แสดงอะไร | เหมาะกับ |
|---|---|---|
| `barbs` | ทิศ + ความเร็วแบบอ่านค่าได้ | แผนที่อากาศมาตรฐาน |
| `quiver` | ทิศ + ความเร็วตามความยาวลูกศร | ภาพรวมแบบเข้าใจง่าย |
| `streamplot` | รูปแบบการไหล | เขตร้อน ดูการพัดเข้าหากัน |

### `barbs` — สัญลักษณ์มาตรฐาน

- ก้านชี้ไป **ทิศที่ลมพัดมาจาก**
- ขีดสั้น = 5 นอต · ขีดยาว = 10 นอต · ธงสามเหลี่ยม = 50 นอต

⚠️ **barb ถูกนิยามด้วยหน่วยนอต** ข้อมูล ERA5 เป็น m/s ต้องคูณ 1.94384 ก่อน
ถ้าลืม จะอ่านค่าผิดไปเกือบเท่าตัว

⚠️ ต้อง **เว้นช่วงกริด** ด้วย `[::step]` ไม่งั้นสัญลักษณ์ทับกันจนอ่านไม่ออก

In [ ]:
u = ds["u"].sel(time="2021-09-24T12:00", level=850)
v = ds["v"].sel(time="2021-09-24T12:00", level=850)

MS_TO_KT = 1.94384
step = 2        # วาดทุก ๆ 2 จุดกริด

fig, axes = plt.subplots(1, 3, figsize=(17, 5),
                         subplot_kw={"projection": ccrs.PlateCarree()})
for ax in axes:
    add_basemap(ax, [98, 110, 10, 22], gridlines=False)

# barbs
axes[0].barbs(u["longitude"].values[::step], u["latitude"].values[::step],
              u.values[::step, ::step] * MS_TO_KT,
              v.values[::step, ::step] * MS_TO_KT,
              length=5.5, linewidth=0.6, transform=ccrs.PlateCarree())
axes[0].set_title("barbs — อ่านค่าความเร็วได้", fontsize=10, loc="left")

# quiver
q = axes[1].quiver(u["longitude"].values[::step], u["latitude"].values[::step],
                   u.values[::step, ::step], v.values[::step, ::step],
                   scale=400, width=0.004, transform=ccrs.PlateCarree())
axes[1].quiverkey(q, 0.88, 1.03, 20, "20 m/s", labelpos="E")
axes[1].set_title("quiver — เข้าใจง่าย เห็นความแรงจากความยาว", fontsize=10, loc="left")

# streamplot
axes[2].streamplot(u["longitude"].values, u["latitude"].values,
                   u.values, v.values, density=1.5, linewidth=0.7,
                   arrowsize=0.9, color="dimgray",
                   transform=ccrs.PlateCarree())
axes[2].set_title("streamplot — เห็นรูปแบบการไหล", fontsize=10, loc="left")

plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### `step` มีผลมาก ลองเทียบดู

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax, s, title in zip(axes, [1, 2, 6],
                        ["step=1 ทึบจนอ่านไม่ออก",
                         "step=2 พอดี",
                         "step=6 ห่างเกินไป เสียรายละเอียด"]):
    add_basemap(ax, [98, 110, 10, 22], gridlines=False)
    ax.barbs(u["longitude"].values[::s], u["latitude"].values[::s],
             u.values[::s, ::s] * MS_TO_KT, v.values[::s, ::s] * MS_TO_KT,
             length=5, linewidth=0.5, transform=ccrs.PlateCarree())
    ax.set_title(title, fontsize=10, loc="left")

plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

**วิธีเลือก step:** อยากให้มีสัญลักษณ์ราว 15–25 ตัวในแนวนอน
ถ้าโดเมนกว้าง 41 จุดกริด ก็ใช้ `step=2` ได้ประมาณ 20 ตัว กำลังดี

## 7. ชั้นที่ 5 — จุด เส้น และข้อความ

ใช้ `plot`, `scatter`, `text` ได้เหมือน matplotlib ปกติ แค่ต้องเติม `transform`

In [ ]:
provinces = {
    "อุบลราชธานี": (15.25, 104.87),
    "มุกดาหาร": (16.54, 104.72),
    "นครราชสีมา": (14.97, 102.08),
    "เชียงใหม่": (18.79, 98.98),
}

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [95, 112, 8, 22])

cs = ax.contour(mslp["longitude"], mslp["latitude"], mslp.values,
                levels=np.arange(980, 1021, 4), colors="gray",
                linewidths=0.8, transform=ccrs.PlateCarree())
ax.clabel(cs, inline=True, fontsize=7, fmt="%d")

for name, (lat, lon) in provinces.items():
    ax.plot(lon, lat, marker="o", color="crimson", markersize=7,
            markeredgecolor="black", transform=ccrs.PlateCarree(), zorder=5)
    ax.text(lon + 0.25, lat + 0.15, name, fontsize=9, color="darkred",
            transform=ccrs.PlateCarree(), zorder=5)

# หาและทำเครื่องหมายศูนย์กลางความกดอากาศต่ำสุด
idx = mslp.argmin(dim=["latitude", "longitude"])
lat_low = float(mslp["latitude"][idx["latitude"]])
lon_low = float(mslp["longitude"][idx["longitude"]])
ax.plot(lon_low, lat_low, marker="x", color="blue", markersize=14,
        markeredgewidth=3, transform=ccrs.PlateCarree(), zorder=6)
ax.text(lon_low + 0.3, lat_low - 0.6, f"L {float(mslp.min()):.0f}",
        color="blue", fontweight="bold", transform=ccrs.PlateCarree(), zorder=6)

ax.set_title("ความกดอากาศผิวพื้นและตำแหน่งจังหวัด\n2021-09-24 12 UTC", loc="left")
plt.show()

print(f"ศูนย์กลางความกดอากาศต่ำสุด {lat_low:.2f}N {lon_low:.2f}E "
      f"= {float(mslp.min()):.1f} hPa")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 8. ซูมเข้าพื้นที่ศึกษา

เปลี่ยนแค่ `set_extent` ก็ซูมได้ ไม่ต้องตัดข้อมูล
แต่ถ้าโดเมนเล็กมาก ควรตัดข้อมูลด้วย `sel(slice(...))` เพื่อให้ `levels` คำนวณจากค่าในพื้นที่จริง

In [ ]:
# ตัดข้อมูลเฉพาะภาคอีสานตอนล่าง
box = dict(longitude=slice(101, 107), latitude=slice(13, 18))
rain_zoom = ds["tp"].sel(time="2021-09-24T12:00").sel(**box)
u_zoom = ds["u"].sel(time="2021-09-24T12:00", level=850).sel(**box)
v_zoom = ds["v"].sel(time="2021-09-24T12:00", level=850).sel(**box)

print("ขนาดข้อมูลหลังตัด:", rain_zoom.shape)

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [101, 107, 13, 18])

cf = ax.contourf(rain_zoom["longitude"], rain_zoom["latitude"], rain_zoom.values,
                 levels=[1, 2, 5, 10, 20, 30, 40], cmap="GnBu", extend="max",
                 transform=ccrs.PlateCarree())
ax.barbs(u_zoom["longitude"].values, u_zoom["latitude"].values,
         u_zoom.values * MS_TO_KT, v_zoom.values * MS_TO_KT,
         length=5.5, linewidth=0.6, transform=ccrs.PlateCarree())

ax.plot(104.87, 15.25, "r^", markersize=10, transform=ccrs.PlateCarree())
ax.text(104.97, 15.35, "อุบลราชธานี", fontsize=10, color="darkred",
        transform=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.4, linestyle="--")
gl.top_labels = gl.right_labels = False

plt.colorbar(cf, ax=ax, shrink=0.8, label="ฝนสะสม 6 ชั่วโมง (mm)")
ax.set_title("ซูมภาคอีสานตอนล่าง — ฝนและลม 850 hPa\n2021-09-24 12 UTC", loc="left")
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 9. หลายแผนที่ในภาพเดียว

ใช้ `subplots` พร้อม `subplot_kw={"projection": ...}` แล้วสั่งงานผ่าน `axes[i]`

**กฎสำคัญ: ทุกช่องต้องใช้ `levels` และ `cmap` ชุดเดียวกัน** ไม่งั้นเปรียบเทียบไม่ได้
และควรใช้ colorbar อันเดียวร่วมกัน

In [ ]:
times = ["2021-09-23T12:00", "2021-09-24T00:00",
         "2021-09-24T12:00", "2021-09-25T00:00"]
levels = np.arange(980, 1021, 4)

fig, axes = plt.subplots(2, 2, figsize=(12, 9),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax, t in zip(axes.flat, times):
    p = ds["msl"].sel(time=t) / 100
    r = ds["tp"].sel(time=t)

    add_basemap(ax, [95, 115, 5, 25], gridlines=False)

    cf = ax.contourf(r["longitude"], r["latitude"], r.values,
                     levels=[1, 2, 5, 10, 20, 40], cmap="GnBu",
                     extend="max", transform=ccrs.PlateCarree())
    cs = ax.contour(p["longitude"], p["latitude"], p.values,
                    levels=levels, colors="black", linewidths=0.6,
                    transform=ccrs.PlateCarree())
    ax.clabel(cs, inline=True, fontsize=6, fmt="%d")
    ax.set_title(t.replace("T", " ") + " UTC", fontsize=10, loc="left")

fig.colorbar(cf, ax=axes, orientation="horizontal", shrink=0.4,
             pad=0.04, label="ฝนสะสม 6 ชั่วโมง (mm)")
fig.suptitle("การเคลื่อนตัวของพายุเตี้ยนหมู่", fontsize=14)
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 10. บันทึกรูปสำหรับรายงาน

```python
fig.savefig("ชื่อไฟล์.png", dpi=300, bbox_inches="tight")
```

| พารามิเตอร์ | ใช้ทำอะไร |
|---|---|
| `dpi=300` | ความละเอียดสำหรับงานตีพิมพ์ (หน้าจอใช้ 100 ก็พอ) |
| `bbox_inches="tight"` | ตัดขอบขาวส่วนเกิน ไม่ให้ชื่อแผนที่โดนตัด |
| `facecolor="white"` | กันพื้นหลังโปร่งใสเวลาแปะลงเอกสาร |

นามสกุล `.pdf` หรือ `.svg` จะได้ภาพเวกเตอร์ ขยายเท่าไรก็ไม่แตก เหมาะกับวารสาร

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
add_basemap(ax, [95, 115, 5, 25])

cf = ax.contourf(wind["longitude"], wind["latitude"], wind.values,
                 levels=np.arange(0, 26, 2), cmap="YlGnBu", extend="max",
                 transform=ccrs.PlateCarree())
plt.colorbar(cf, ax=ax, shrink=0.8, label="ความเร็วลม 850 hPa (m s$^{-1}$)")
ax.set_title("ตัวอย่างรูปสำหรับรายงาน", loc="left")

fig.savefig("my_weather_map.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
!ls -lh my_weather_map.png

## 11. ข้อผิดพลาดที่เจอบ่อย

| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| ข้อมูลไปกองผิดที่ | ลืม `transform=` | ใส่ `transform=ccrs.PlateCarree()` ทุกคำสั่งพล็อต |
| แผนที่ว่างเปล่า ไม่มีข้อมูล | `set_extent` อยู่นอกโดเมนข้อมูล | ตรวจช่วง lon/lat ของข้อมูลก่อน |
| `sel(slice(5, 21))` ได้ข้อมูลเปล่า | latitude เรียงจากมากไปน้อย | `ds.sortby("latitude")` ก่อน |
| barb อ่านค่าได้ผิด | ใส่ m/s แทนนอต | คูณ 1.94384 ก่อน |
| สัญลักษณ์ลมทับกันดำพรืด | ไม่ได้เว้นช่วงกริด | ใช้ `[::step]` |
| สีพื้นทับข้อมูล | `LAND`/`OCEAN` ไม่ได้ตั้ง zorder | ใส่ `zorder=0` |
| ชื่อแผนที่โดนตัดตอน save | ไม่ได้ใส่ `bbox_inches` | `savefig(..., bbox_inches="tight")` |
| ตัวอักษรไทยเป็นสี่เหลี่ยม □□□ | ยังไม่ได้ตั้งฟอนต์ไทย | รันเซลล์ตั้งฟอนต์ตอนต้นบท |

### เช็กลิสต์ก่อนส่งแผนที่

- [ ] มีชื่อแผนที่ บอกว่าแสดงอะไร **เวลาไหน**
- [ ] colorbar มีหน่วยกำกับ
- [ ] มีเส้นชายฝั่งหรือพรมแดนให้รู้ว่าที่ไหน
- [ ] มีเส้นกริดพร้อมตัวเลขพิกัด
- [ ] ระบุแหล่งข้อมูล (ERA5, กรมอุตุฯ)
- [ ] ถ้ามีหลายช่อง ใช้ระดับสีชุดเดียวกัน

## 12. แบบฝึกหัดท้ายบท

1. สร้างแผนที่ **อุณหภูมิ 2 เมตร** เวลา `2021-09-20T06:00` ครอบคลุมประเทศไทย
   (lon 96–106, lat 5–21) พร้อมเส้นชายฝั่ง พรมแดน gridline colorbar และชื่อแผนที่ครบ
2. สร้างแผนที่ **ความสูง geopotential ที่ 500 hPa** (หน่วย gpm = `z` หาร 9.80665)
   เป็นเส้นชั้นสีแดง ซ้อนบนแรเงาความเร็วลมที่ 500 hPa
3. เปรียบเทียบ **ฝนสะสม 6 ชั่วโมง** 2 ช่วงเวลาข้างกัน ใช้ `levels` และ `cmap` ชุดเดียวกัน
   พร้อม colorbar อันเดียวร่วมกัน
4. สร้างแผนที่ที่มีครบทั้ง 5 ชั้น — แผนที่ฐาน + แรเงา + เส้นชั้น + ลม + จุดจังหวัด
   แล้ว save เป็น PNG 300 dpi

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3

In [ ]:
# ข้อ 4

---

### เฉลยแบบฝึกหัด

ผู้สอนจะแจกเฉลยหลังหมดเวลาทำแบบฝึกหัดในคาบ

> ลองด้วยตัวเองให้สุดก่อน การติดแล้วแก้ได้เองคือช่วงที่สมองเรียนรู้มากที่สุด
> ถ้าติดเกิน 10 นาทีให้ยกมือถามได้เลย ไม่ต้องรอ


---

## ไปต่อที่บทไหน

พร้อมแล้วสำหรับ **บทที่ 12 — วิเคราะห์พายุที่ส่งผลต่อประเทศไทย**
ซึ่งใช้ฟังก์ชันสำเร็จรูปที่ห่อขั้นตอนในบทนี้เอาไว้ จะได้ไม่ต้องพิมพ์ซ้ำทุกครั้ง

ลองเปิด `utils/era5_utils.py` อ่านดูได้เลย — ทุกบรรทัดในนั้นคือสิ่งที่เรียนในบทนี้
`eu.shade()` ก็คือ `contourf` ที่ใส่ `transform` ให้แล้ว
`eu.barbs()` ก็คือ `barbs` ที่แปลงหน่วยเป็นนอตและเว้นช่วงกริดให้แล้ว

**การอ่านโค้ดคนอื่นออกคือทักษะที่สำคัญไม่แพ้การเขียนเอง**